In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.covariance import LedoitWolf
from itertools import product


import config.settings as settings
import src.produccion_x_equipo as crea_produccion
import src.extract_files as extract
import src.filter as filtra
import src.fc_prophet as fcProphet
#import src.produccion_x_equipo as produccion_x_equipo
import importlib
importlib.reload(filtra)
importlib.reload(fcProphet)
#import src.notebooks.TestFunctions as testfunctions
%load_ext autoreload
%autoreload 2


x is: 1


/home/jorge/Projects/Radioterapia/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [22]:
input_path = settings.DATA_DIR / "processed" / "demanda" / "demanda_diaria_total.csv"
path_fc_train_total = settings.DATA_DIR / "processed" /"forecast" / "fcProphet_top_level_pred_actual_res.csv"
path_fc_train_total_ex = settings.DATA_DIR / "processed" /"forecast" / "fcProphet_top_level_pred_actual_res_ex.csv"
df_prod_total = pd.read_csv(input_path)
df_fc_train_total = pd.read_csv(path_fc_train_total)

In [12]:
print(df_prod_total.head(2))
print(df_prod_total["ds"].min())
print(df_prod_total["ds"].max())

           ds    y  capacidad  festivo  fin_de_semana  control_de_calidad  \
0  2022-05-03  407        564    False          False               False   
1  2022-05-04  443        564    False          False               False   

   falla  mantenimiento  
0  False          False  
1  False          False  
2022-05-03
2026-04-30


In [13]:
print(df_fc_train_total.head(2))
print(df_fc_train_total["ds"].min())
print(df_fc_train_total["ds"].max())

           ds    y        yhat  residuals
0  2022-05-03  407  452.971519 -45.971519
1  2022-05-04  443  454.581204 -11.581204
2022-05-03
2026-03-01


In [17]:
df_fc_train_total["ds"]  = pd.to_datetime(df_fc_train_total["ds"] )
df_prod_total["ds"] = pd.to_datetime(df_prod_total["ds"] )

In [18]:
df_fc_train_total_ex = df_fc_train_total.merge(
    df_prod_total.drop(columns = "y"),
    on = "ds",
    how = "left",
    validate = "many_to_one"
)

In [21]:
print(df_fc_train_total_ex.head(3))

          ds    y        yhat  residuals  capacidad  festivo  fin_de_semana  \
0 2022-05-03  407  452.971519 -45.971519        564    False          False   
1 2022-05-04  443  454.581204 -11.581204        564    False          False   
2 2022-05-05  451  450.120904   0.879096        564    False          False   

   control_de_calidad  falla  mantenimiento  
0               False  False          False  
1               False  False          False  
2               False  False          False  


In [24]:
df_fc_train_total_ex.to_csv(path_fc_train_total_ex, index = False)